> 在线讲义：[第 21 章 · 拼装：一个 Transformer 层；encoder / decoder 全图；三种 attention](https://wikieden.github.io/transformer-visual/lessons/21-transformer-layer.html) · 本书《从零读懂 Transformer》

# 第 21 章 · 拼装：一个 Transformer 层；encoder / decoder 全图；三种 attention

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/21-transformer-layer.html) · 仓库里 `transformer-deep-dive/notes/lessons/21-transformer-layer.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| encoder 层 | $\text{LN}(x + \text{MHA}(x))$ → $\text{LN}(z + \text{FFN}(z))$ | `EncoderLayer` | 2 个 LayerNorm；(1, 3, 8) → (1, 3, 8) |
| decoder 层 | 多一道"查原文" | `DecoderLayer` | 3 个 LayerNorm；(1, 2, 8) → (1, 2, 8) |
| ① encoder self | 原文问原文 | `self.attn(x, x, x)` | 分数表 3 × 3 |
| ② masked self | 译文问译文，遮右上 | `attn_mask=causal(m)` | 2 × 2，右上角权重 0 |
| ③ encoder-decoder | Q 是译文，K、V 是 e | `self.cross_attn(a, e, e)` | 2 × 3 |
| 叠 6 层 | 形状不变 | `nn.ModuleList` | 还是 (1, 3, 8) |
| 逐词生成 | 第 k 轮查原文 | 循环 k = 1…4 | 分数表 1 × 3、2 × 3、3 × 3、4 × 3 |

用小尺寸 d_model = 8、h = 2、d_ff = 32，随机权重（没训练），只看形状和结构。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
from torch import nn
D, H, D_FF = 8, 2, 32

**函数 `causal`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def causal(n: int) -> torch.Tensor:
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

**类 `EncoderLayer`**：两道工序，post-LN：z = LN(x + MHA(x))，out = LN(z + FFN(z))。

In [ ]:
class EncoderLayer(nn.Module):
    """两道工序，post-LN：z = LN(x + MHA(x))，out = LN(z + FFN(z))。"""

    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(D, H, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(D, D_FF), nn.ReLU(), nn.Linear(D_FF, D))
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)

    def forward(self, x):
        a, w = self.attn(x, x, x)
        z = self.ln1(x + a)
        return self.ln2(z + self.ffn(z)), w

**类 `DecoderLayer`**：三道工序：看自己写过的（遮）→ 回头查原文（Q 是译文，K、V 是 e）→ 消化。

In [ ]:
class DecoderLayer(nn.Module):
    """三道工序：看自己写过的（遮）→ 回头查原文（Q 是译文，K、V 是 e）→ 消化。"""

    def __init__(self):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(D, H, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(D, H, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(D, D_FF), nn.ReLU(), nn.Linear(D_FF, D))
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(D), nn.LayerNorm(D), nn.LayerNorm(D)

    def forward(self, y, e):
        m = y.shape[1]
        a1, w_self = self.self_attn(y, y, y, attn_mask=causal(m))
        a = self.ln1(y + a1)
        a2, w_cross = self.cross_attn(a, e, e)  # Q = a，K = V = e
        b = self.ln2(a + a2)
        return self.ln3(b + self.ffn(b)), w_self, w_cross

**函数 `count`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def count(m: nn.Module, kind: type) -> int:
    return sum(isinstance(c, kind) for c in m.modules())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
torch.manual_seed(0)
enc, dec = EncoderLayer().eval(), DecoderLayer().eval()
src = torch.randn(1, 3, D)  # 原文"狗咬人"：1 句 × 3 个词 × d_model
tgt = torch.randn(1, 2, D)  # 已写的 "dog bites"
with torch.no_grad():
    e, w_enc = enc(src)
    out, w_self, w_cross = dec(tgt, e)
print("LayerNorm 个数：encoder 层", count(enc, nn.LayerNorm), "  decoder 层", count(dec, nn.LayerNorm))
print(
    "形状始终不变：encoder",
    tuple(src.shape),
    "→",
    tuple(e.shape),
    "；decoder",
    tuple(tgt.shape),
    "→",
    tuple(out.shape),
)

In [ ]:
print("\n三种 attention 的分数表（权重）形状 = 提问的词数 × 被问的词数")
print("  ① encoder self   :", tuple(w_enc.shape[1:]))
print("  ② masked self    :", tuple(w_self.shape[1:]), " 右上角权重:", float(w_self[0, 0, 1]))
print("  ③ encoder-decoder:", tuple(w_cross.shape[1:]))

In [ ]:
print("\n叠 6 层，形状还是不变")
encs = nn.ModuleList(EncoderLayer() for _ in range(6)).eval()
with torch.no_grad():
    x = src
    for layer in encs:
        x, _ = layer(x)
print("  ", tuple(x.shape))

In [ ]:
print("\n逐词生成：encoder 只跑一次，decoder 第 k 轮的'查原文'分数表是 k × 3")
with torch.no_grad():
    e, _ = enc(src)
    for k in range(1, 5):
        y = torch.randn(1, k, D)  # 已写的 k 个词（这里用随机向量代替）
        _, _, w_cross = dec(y, e)
        print(f"  第 {k} 轮: 输入 {k} 个词，分数表 {tuple(w_cross.shape[1:])}，取最后一行去选下一个词")
print(
    "\nPyTorch 自带的 nn.TransformerEncoderLayer 默认也是 post-LN（norm_first=False）:",
    nn.TransformerEncoderLayer(D, H, D_FF).norm_first is False,
)

## 3. 代码里才看得到的

1. **`nn.MultiheadAttention(q, k, v)` 的三个参数就是"谁问、问谁"**：self-attention 三个都传同一个；cross-attention 传 `(a, e, e)`。它同时返回各头平均后的权重表，形状正好是"提问的词数 × 被问的词数"。
2. **`batch_first=True`**：输入形状是"几句 × 几个词 × 每词几个数"。不写的话默认把"几个词"放在最前面，形状对不上也可能不报错。
3. **`.eval()`**：关掉 dropout（训练时随机丢掉一部分数，防止死记硬背；续篇讲）。只看形状和结构时要关掉，结果才稳定。
4. **PyTorch 自带 `nn.TransformerEncoderLayer`**，默认 `norm_first=False`，也就是原论文的 post-LN；改成 True 就是 pre-LN。

## 4. 自己改一改

- 把原文改成 5 个词、译文 4 个词（第 21 章题 3），三张分数表各是几行几列？先说答案再跑。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 编码一层两道工，开会消化各一回；每道之后加回原稿、调范围，进出 512 能叠六层。
> 解码一层三道工，看自己（要遮）、查原文（不遮）、再消化；三次 LayerNorm 数工序。
> 分数表：行看谁提问，列看问了谁；输出几行跟提问的人走。
> 只有"还没写出来"才要遮：原文早给全，译文后面空。
> 原文读一次存成 e，译文一轮写一个，取最后一行选下一个，写出结束就收工。